In [1]:
import alexapy

In [17]:
import numpy as np
import librosa

class SignalProcessor:
    """
    Handles transient detection (counting taps) and feature extraction
    (generating a sonic fingerprint) for incoming audio buffers.
    """
    def __init__(self, sample_rate: int = 16000, hop_length: int = 512):
        self.sample_rate = sample_rate
        self.hop_length = hop_length

    def calculate_energy(self, audio_frame: np.ndarray) -> float:
        """
        Calculates the Root Mean Square (RMS) energy of a raw audio frame.
        Used by the listener thread to detect if a sound broke the noise floor.
        """
        if len(audio_frame) == 0:
            return 0.0
        return float(np.sqrt(np.mean(audio_frame**2)))

    def count_taps(self, audio_clip: np.ndarray, debounce_seconds: float = 0.05) -> int:
        """
        Detects sudden spikes in acoustic energy to count discrete impacts.
        Includes a debounce parameter to prevent room echo or muddy strikes 
        from being counted as multiple taps.
        """
        # Convert the debounce time (seconds) into the number of frames required by librosa
        wait_frames = int((debounce_seconds * self.sample_rate) / self.hop_length)

        onsets = librosa.onset.onset_detect(
            y=audio_clip,
            sr=self.sample_rate,
            hop_length=self.hop_length,
            wait=wait_frames,
            backtrack=False
        )
        print(onsets)
        return len(onsets)

    def extract_features(self, audio_clip: np.ndarray) -> np.ndarray:
        """
        Compresses a 1.5-second audio array into a flat, 1D feature vector
        representing the acoustic profile of the tap.
        """
        # 1. Mel-Frequency Cepstral Coefficients (Timbre/Material of the tap)
        # 13 coefficients is standard for non-speech audio classification
        mfccs = librosa.feature.mfcc(y=audio_clip, sr=self.sample_rate, n_mfcc=13)
        
        # 2. Spectral Centroid (Brightness/Sharpness of the tap)
        centroid = librosa.feature.spectral_centroid(y=audio_clip, sr=self.sample_rate)
        
        # 3. Spectral Rolloff (High-frequency decay)
        rolloff = librosa.feature.spectral_rolloff(y=audio_clip, sr=self.sample_rate)
        
        # Compress the 2D time-series arrays into flat 1D statistical summaries
        # We take the mean and standard deviation to capture the 'shape' of the sound over time
        feature_vector = np.hstack((
            np.mean(mfccs, axis=1), np.std(mfccs, axis=1),
            np.mean(centroid), np.std(centroid),
            np.mean(rolloff), np.std(rolloff)
        ))
        
        return feature_vector

In [18]:
import pyaudio
import numpy as np
from collections import deque

class AudioListener:
    def __init__(self, processor, threshold=0.015, pre_roll_sec=0.2, record_sec=1.0):
        """
        Initializes the microphone stream.
        - threshold: The RMS energy level required to trigger a capture.
        - pre_roll_sec: How much audio to keep *before* the trigger (captures the initial transient).
        - record_sec: How much audio to record *after* the trigger.
        """
        self.processor = processor
        self.threshold = threshold
        self.chunk_size = 1024
        
        # PyAudio configuration
        self.format = pyaudio.paFloat32  # Float32 is required by librosa
        self.channels = 1                # Mono audio
        self.rate = self.processor.sample_rate

        # Calculate how many 1024-frame chunks make up our time windows
        self.pre_roll_chunks = int((pre_roll_sec * self.rate) / self.chunk_size)
        self.record_chunks = int((record_sec * self.rate) / self.chunk_size)
        
        # The Ring Buffer
        self.buffer = deque(maxlen=self.pre_roll_chunks)
        self.audio = pyaudio.PyAudio()

    def listen(self, trigger_callback):
        """
        Opens the mic and starts the infinite listening loop.
        When a tap is detected, it processes the sound and passes the 
        results to the provided callback function.
        """
        stream = self.audio.open(format=self.format,
                                 channels=self.channels,
                                 rate=self.rate,
                                 input=True,
                                 frames_per_buffer=self.chunk_size)
        
        print(f"[*] Mic active. Listening for taps (Threshold: {self.threshold})...")
        
        try:
            while True:
                # Read a chunk from the mic
                data = stream.read(self.chunk_size, exception_on_overflow=False)
                
                # Convert raw bytes to a numpy float array
                audio_frame = np.frombuffer(data, dtype=np.float32)
                
                # Push into the ring buffer (oldest chunk falls off automatically)
                self.buffer.append(audio_frame)
                
                # The Energy Gate
                if self.processor.calculate_energy(audio_frame) > self.threshold:
                    self._capture_and_process(stream, trigger_callback)
                    
        except KeyboardInterrupt:
            print("\n[*] Stopping listener...")
        finally:
            stream.stop_stream()
            stream.close()
            self.audio.terminate()

    def _capture_and_process(self, stream, trigger_callback):
        """
        Called when the threshold is broken. Captures the window,
        processes it, and prevents double-triggering.
        """
        # 1. Grab the pre-roll (everything right before the loud noise)
        frames = list(self.buffer) 
        
        # 2. Record the post-trigger window
        for _ in range(self.record_chunks):
            data = stream.read(self.chunk_size, exception_on_overflow=False)
            frames.append(np.frombuffer(data, dtype=np.float32))
            
        # 3. Flatten the chunks into a single 1D audio array
        full_clip = np.concatenate(frames)
        
        # 4. Pass through the SignalProcessor
        tap_count = self.processor.count_taps(full_clip)
        features = self.processor.extract_features(full_clip)
        
        # 5. Send data back to the Controller thread
        trigger_callback(features, tap_count)
        
        # 6. Flush the buffer so the tail end of the tap doesn't trigger it again instantly
        self.buffer.clear()
    
    
    def test_file(self, file_path: str):
        """
        Loads an audio file, simulates the energy gate by trimming silence,
        and extracts features just like the live microphone listener.
        """
        print(f"[*] Loading test file: {file_path}")
        
        try:
            # Load audio. sr=processor.sample_rate ensures it matches the live mic format
            y, sr = librosa.load(file_path, sr=self.processor.sample_rate, mono=True)
            
            # Simulate the energy gate: Trim leading and trailing silence (below 20dB)
            y_trimmed, index = librosa.effects.trim(y, top_db=20)
            
            print(f"[*] Audio loaded. Original duration: {len(y)/sr:.2f}s, Trimmed: {len(y_trimmed)/sr:.2f}s")
            
            # Pass through the existing SignalProcessor
            tap_count = self.processor.count_taps(y_trimmed)
            features = self.processor.extract_features(y_trimmed)
            
            print(f"[*] Taps detected: {tap_count}")
            print(f"[*] Feature vector generated (Length: {len(features)})")
            
            return features, tap_count
            
        except Exception as e:
            print(f"[!] Error loading file {file_path}: {e}")
            return None, 0

In [19]:
processor = SignalProcessor()
listener = AudioListener(processor=processor)

In [22]:
listener.test_file("tests/file 2.m4a")

[*] Loading test file: tests/file 2.m4a
[*] Audio loaded. Original duration: 12.58s, Trimmed: 5.25s
[  3  52  59  96 103 109 150 157]
[*] Taps detected: 8
[*] Feature vector generated (Length: 30)


/var/folders/9h/4775nyxn5qn0zy8v_5c8ky8r0000gn/T/ipykernel_41399/3153610590.py:102: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=self.processor.sample_rate, mono=True)


(array([-4.75139740e+02,  1.31745499e+02, -1.45889645e+01,  1.17768478e+01,
         4.05952513e-01,  3.75008368e+00,  1.94203508e+00, -2.20686483e+00,
        -5.58192635e+00,  3.20431471e+00,  6.67880058e-01,  3.22045684e+00,
        -6.72607481e-01,  8.75563660e+01,  3.11866226e+01,  3.22517891e+01,
         7.89777517e+00,  7.15245676e+00,  1.31537113e+01,  9.80819416e+00,
         7.60478401e+00,  6.11673927e+00,  5.91513252e+00,  4.83587599e+00,
         4.45594835e+00,  4.22413445e+00,  1.31253149e+03,  2.31780201e+02,
         2.72831439e+03,  7.97975951e+02]),
 8)